# Voice2Reel 09 — Colab All-in-One
**دفتر واحد فقط:** تجهيز البيئة → اختيار الشخصية → توليد البيانات → فحصها → تدريب Qwen → توليد المشاهد → الصوت → الفيديو → حفظ النتائج.

**مهم:** اختَر GPU من Runtime → Change runtime type. شغّل الخلايا بالترتيب. التدريب الطويل قد ينقطع في Colab؛ احفظ الملفات على Drive. **لا تضغط Run all قبل مراجعة اختيار الشخصية وإعدادات التدريب.**

**القيود:** الـ200 ألف مثال اصطناعي وليست 200 ألف مشهد مراجَع فنيًا. التدريب الفعلي يحتاج GPU، والـTTS يحتاج إنترنت. فيديو الإنتاج الحالي 540×960، ومزامنة الكلام مع كل مشهد ليست دقيقة بعد.

## 1 — الإعدادات (غيّر هذه الخلية فقط غالبًا)

In [ ]:
#@title ⚙️ إعدادات المشروع
BRANCH = "test-new-data" #@param {type:"string"}
CHARACTER = "classic" #@param ["classic","bold","neon","robot","sketch","ribbon","minimal","professor","astro","shadow","pixel","ink"]
IDEA = "Explain Git branches to beginners" #@param {type:"string"}
LANGUAGE = "en" #@param ["en","ar"]
DATASET_COUNT = 1000 #@param {type:"integer"}
EPOCHS = 3 #@param {type:"integer"}
PATIENCE = 2 #@param {type:"integer"}
RUN_TRAINING = False #@param {type:"boolean"}
RUN_TTS = False #@param {type:"boolean"}
RUN_RENDER = True #@param {type:"boolean"}
USE_DRIVE = True #@param {type:"boolean"}
# 1000 مثال كبداية لفحص المسار؛ بعد نجاح التجربة غيّر إلى 200000.
assert 1 <= DATASET_COUNT <= 2000000
assert 1 <= EPOCHS <= 50
assert 1 <= PATIENCE <= 10
print("الإعدادات جاهزة. التدريب:", RUN_TRAINING, "عدد الأمثلة:", DATASET_COUNT)

## 2 — ربط Google Drive وتجهيز مجلد دائم

In [ ]:
from pathlib import Path
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    STORE = Path('/content/drive/MyDrive/Voice2Reel_09')
else:
    STORE = Path('/content/Voice2Reel_09')
STORE.mkdir(parents=True, exist_ok=True)
DATA = STORE / "dataset"
MODEL_DIR = STORE / "qwen_adapter_run"
RESULTS = STORE / "results"
RESULTS.mkdir(parents=True, exist_ok=True)
print("مسار التخزين:", STORE)

## 3 — تنزيل الفرع وتثبيت المتطلبات
إذا انقطع الاتصال أثناء التنزيل، أعد تشغيل هذه الخلية. لا تمسح Drive.

In [ ]:
import os, subprocess, sys
from pathlib import Path
REPO = Path('/content/Voice2Reel')
def run(*args, cwd=None, timeout=None):
    print("▶", " ".join(map(str,args)), flush=True)
    subprocess.run(list(map(str,args)), cwd=str(cwd) if cwd else None, check=True, timeout=timeout)
if not (REPO / ".git").exists():
    run("git","clone","--depth","1","--branch",BRANCH,"https://github.com/garaahmad/Voice2Reel.git",REPO)
else:
    run("git","fetch","origin",BRANCH,cwd=REPO)
    run("git","checkout",BRANCH,cwd=REPO)
    run("git","pull","--ff-only","origin",BRANCH,cwd=REPO)
run(sys.executable,"-m","pip","install","-q","-r","requirements/character-lab.txt",cwd=REPO)
run(sys.executable,"-m","pip","install","-q","-e",".",cwd=REPO)
run(sys.executable,"-m","pip","install","-q","edge-tts==7.3.1",cwd=REPO)
print("✅ المشروع جاهز",REPO)

## 4 — عرض الشخصية ثم تثبيتها
راجع المعاينة قبل تنفيذ خلية التثبيت التالية. تغيير الشخصية لاحقًا يتطلب إنشاء قفل جديد.

In [ ]:
import sys
sys.path.insert(0,str(REPO))
from IPython.display import display
from voice2reel.character_templates import render_preview, TEMPLATE_IDS
assert CHARACTER in TEMPLATE_IDS
display(render_preview(CHARACTER))
print("الشخصية المحددة:",CHARACTER)

In [ ]:
from voice2reel.character_templates import lock_template, load_lock
LOCK = RESULTS / "character_lock.json"
lock_template(CHARACTER, LOCK)
assert load_lock(LOCK)["template_id"] == CHARACTER
print("🔒 تم تثبيت الشخصية:", CHARACTER, "في", LOCK)

## 5 — توليد بيانات التدريب والتحقق
يتم إنشاء البيانات على Drive. ابدأ بـ1000 مثال، وبعد نجاح الفحص ارفع العدد إلى 200000. **التوليد لا يعني أن الجودة الفنية قد تم التحقق منها.**

In [ ]:
#@title 5 — توليد بيانات التدريب (مع عرض الخطأ الحقيقي)
import os, sys, json, subprocess, traceback
from pathlib import Path

DATA.mkdir(parents=True, exist_ok=True)
cmd = [
    sys.executable, "-m", "scripts.build_large_scene_curriculum",
    "--count", str(DATASET_COUNT),
    "--output", str(DATA),
    "--check-every", str(max(1, min(10000, DATASET_COUNT))),
]
env = os.environ.copy()
env["PYTHONPATH"] = str(REPO) + os.pathsep + env.get("PYTHONPATH", "")
print("▶", " ".join(cmd), flush=True)
print("📁 Project:", REPO, "| Output:", DATA, flush=True)
result = subprocess.run(cmd, cwd=str(REPO), env=env, text=True,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(result.stdout or "(no output)", flush=True)
if result.returncode != 0:
    raise RuntimeError(
        f"فشل توليد البيانات (exit={result.returncode}). "
        "السبب التفصيلي موجود في النص المطبوع فوق. "
        "إذا كان الخطأ ModuleNotFoundError أو ValidationError أرسل النص كاملًا."
    )
manifest_path = DATA / "manifest.json"
if not manifest_path.is_file():
    raise FileNotFoundError(f"ملف النتائج غير موجود: {manifest_path}")
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
print("✅ عدد السجلات:", manifest["splits"])
assert sum(manifest["splits"].values()) == DATASET_COUNT
for name in ("train", "validation", "test"):
    p = DATA / f"{name}.jsonl"
    if not p.is_file():
        raise FileNotFoundError(p)
    print(name, "MB", round(p.stat().st_size / 1e6, 2))


## 6 — فحص عينة من البيانات واختبارات السلامة

In [ ]:
from voice2reel.python_scene_dsl import parse_python_scene
import itertools
for split in ("train","validation","test"):
    with (DATA/f"{split}.jsonl").open(encoding="utf-8") as fh:
        for line in itertools.islice(fh, 30):
            row=json.loads(line)
            plan=parse_python_scene(row["messages"][-1]["content"])
            assert len(plan.shots)>0
print("✅ عينات البيانات صالحة بنيويًا")
run(sys.executable,"-m","pytest","-q",
    "tests/test_python_scene_pipeline.py",
    "tests/test_large_scene_curriculum.py",cwd=REPO)

## 7 — تدريب Qwen (اختياري)
لتشغيل التدريب: فعّل RUN_TRAINING في الخلية الأولى وأعد تشغيل هذه الخلية. يوصى بـ3 Epochs أولًا؛ 50 حد أقصى للتجربة فقط. **لا يوجد استئناف تلقائي للـoptimizer بعد انقطاع Colab** في سكربت التدريب الحالي؛ احتفظ بالـAdapter الناتج.

In [ ]:
#@title 7 — Qwen training (GPU + complete error log)
import os
import sys
import subprocess
import traceback
from pathlib import Path
import torch

REPO = Path("/content/Voice2Reel")
DATA = Path(DATA)
MODEL_DIR = Path(MODEL_DIR)
print("🎮 GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Not available", flush=True)

if not RUN_TRAINING:
    print("⏭ التدريب متوقف. فعّل RUN_TRAINING في خلية الإعدادات أولًا.")
else:
    if not torch.cuda.is_available():
        raise RuntimeError("فعّل GPU من Runtime > Change runtime type")
    if not (REPO / ".git").is_dir():
        raise FileNotFoundError(f"Repository not found: {REPO}")
    for split in ("train", "validation"):
        p = DATA / f"{split}.jsonl"
        if not p.is_file() or p.stat().st_size == 0:
            raise FileNotFoundError(f"Missing or empty dataset: {p}")

    print("🔄 Updating project...", flush=True)
    subprocess.run(["git", "pull", "--ff-only", "origin", "test-new-data"],
                   cwd=str(REPO), check=True)
    subprocess.run([sys.executable, "-m", "py_compile",
                    str(REPO / "voice2reel/train_python_scene.py")],
                   check=True)

    print("📦 Checking training dependencies...", flush=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "-r", "requirements/text.txt"], cwd=str(REPO), check=True)

    # Do not overwrite existing checkpoints or previous attempts.
    output = MODEL_DIR
    if output.exists() and any(output.iterdir()):
        index = 2
        while True:
            candidate = MODEL_DIR.with_name(f"{MODEL_DIR.name}_{index}")
            if not candidate.exists() or not any(candidate.iterdir()):
                output = candidate
                break
            index += 1
        print("⚠️ Existing training files preserved. New run:", output, flush=True)

    output.mkdir(parents=True, exist_ok=True)
    log_path = output / "training.log"
    cmd = [
        sys.executable, "-u", "-m", "voice2reel.train_python_scene",
        "--train", str(DATA / "train.jsonl"),
        "--validation", str(DATA / "validation.jsonl"),
        "--output", str(output),
        "--epochs", str(EPOCHS),
        "--patience", str(PATIENCE)
    ]
    env = os.environ.copy()
    env["PYTHONPATH"] = str(REPO) + os.pathsep + env.get("PYTHONPATH", "")
    env["PYTHONUNBUFFERED"] = "1"
    print("🚀 Starting Qwen training...", flush=True)
    print("📝 Log:", log_path, flush=True)

    # Keep the complete traceback in Drive; show output live in Colab.
    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(
            cmd, cwd=str(REPO), env=env,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1
        )
        try:
            for line in process.stdout:
                log.write(line)
                log.flush()
                print(line, end="", flush=True)
            return_code = process.wait()
        except KeyboardInterrupt:
            process.terminate()
            process.wait()
            raise

    if return_code != 0:
        lines = log_path.read_text(encoding="utf-8", errors="replace").splitlines()
        print("\n❌ Training failed. Last 40 log lines:\n", flush=True)
        print("\n".join(lines[-40:]), flush=True)
        print("📄 Full error log:", log_path, flush=True)
        raise RuntimeError(f"Qwen training failed (exit code {return_code}). See log above.")
    print("✅ Training completed:", output, flush=True)
    if output != MODEL_DIR:
        print("⚠️ Adapter location changed. Update MODEL_DIR to:", output)


## 8 — إنشاء مشاهد بواسطة Qwen أو خطة تجريبية
إذا كان الـAdapter موجودًا، سيُستخدم Qwen. بدونه نُنشئ خطة تجريبية واضحة لتجربة الصوت والرندر؛ **هذا ليس ناتج Qwen**.

In [ ]:
from voice2reel.production_contract import ProductionPlan
PLAN_PATH=RESULTS/"scene.json"
ADAPTER=MODEL_DIR/"adapter"
if ADAPTER.is_dir():
    run(sys.executable,"-m","voice2reel.generate_python_scene",
        "--idea",IDEA,"--lock",LOCK,"--adapter",ADAPTER,
        "--output",PLAN_PATH,cwd=REPO)
    print("🧠 مشاهد مولّدة بواسطة Qwen")
else:
    is_ar=LANGUAGE=="ar"
    shots=[
      dict(caption="المشكلة" if is_ar else "The problem",
           narration="كيف نحل هذه المشكلة بخطوات بسيطة؟" if is_ar else "Let's understand the problem step by step.",
           seconds=4.0,gesture="think",camera="close",transition="cut"),
      dict(caption="الحل" if is_ar else "The solution",
           narration="نقسم المهمة إلى خطوات واضحة وسهلة." if is_ar else "Break the task into clear and practical steps.",
           seconds=5.0,gesture="explain",camera="medium",transition="fade"),
      dict(caption="جرّب بنفسك" if is_ar else "Try it",
           narration="الآن جرب الفكرة بنفسك." if is_ar else "Now try the idea in your own project.",
           seconds=4.0,gesture="celebrate",camera="wide",transition="fade"),
    ]
    plan=ProductionPlan.model_validate(dict(title=IDEA[:100],shots=shots))
    PLAN_PATH.write_text(plan.model_dump_json(indent=2),encoding="utf-8")
    print("🧪 خطة اختبار يدوية (Qwen لم يتدرب بعد)")
plan=ProductionPlan.model_validate_json(PLAN_PATH.read_text(encoding="utf-8"))
print("المشاهد:",len(plan.shots),"المدة:",sum(x.seconds for x in plan.shots),"ثانية")

## 9 — توليد التعليق الصوتي (اختياري)
يتطلب اتصال إنترنت. النص يُرسل إلى خدمة Microsoft عبر edge-tts. الصوت الحالي ليس متزامنًا بدقة مع المشاهد.

In [ ]:
VOICE=RESULTS/"voice.mp3"
if RUN_TTS:
    run(sys.executable,"-m","voice2reel.production_voice",
        "--plan",PLAN_PATH,"--output",VOICE,"--language",LANGUAGE,cwd=REPO)
    print("🔊",VOICE)
else:
    print("⏭ الصوت متوقف. سيُنتج الفيديو بدون تعليق صوتي.")

## 10 — إنتاج الفيديو ودمج الموسيقى
للموسيقى، ارفع ملفًا مرخّصًا إلى Drive داخل مجلد النتائج باسم **music.mp3**. عند غياب الملف يُنتج الفيديو بدون موسيقى. لا يُستخدم صوت قديم إلا عند تفعيل RUN_TTS.

In [ ]:
VIDEO=RESULTS/"final.mp4"
MUSIC=RESULTS/"music.mp3"
if RUN_RENDER:
    args=[sys.executable,"-m","voice2reel.production_pipeline",
          "--plan",PLAN_PATH,"--lock",LOCK,"--output",VIDEO]
    if RUN_TTS and VOICE.exists(): args+=["--voice",VOICE]
    if MUSIC.is_file(): args+=["--music",MUSIC]
    run(*args,cwd=REPO,timeout=1800)
    assert VIDEO.is_file() and VIDEO.stat().st_size>0
    print("🎬 تم إنشاء:",VIDEO,"الحجم MB:",round(VIDEO.stat().st_size/1e6,2))
else:
    print("⏭ الرندر متوقف")

## 11 — مشاهدة الفيديو وحفظ النتائج
المخرجات محفوظة في Drive عند USE_DRIVE=True. إذا واجهت خطأ، أرسل **نص الخطأ كاملًا ورقم الخلية**.

In [ ]:
from IPython.display import Video, display
if VIDEO.is_file():
    display(Video(str(VIDEO),embed=False,width=300))
    print("الفيديو:",VIDEO)
else:
    print("لم يتم إنشاء فيديو بعد.")
print("البيانات:",DATA)
print("الـAdapter:",ADAPTER)
print("المخرجات:",RESULTS)